
# ПЗ-8. Проектирование архитектуры Big Data и выбор промышленного конвейера обработки

**Дисциплина:** Системы обработки больших данных  
**Тема практического занятия:** Выбор архитектуры обработки, оценка SLA, сравнение batch / micro-batch / stream  
**Формат работы:** индивидуальный вариант  
**Важно:** ноутбук и все датасеты должны лежать в одной папке.

---

## Цель практической работы

На основе индивидуального набора событий:
1. исследовать профиль нагрузки и характер поступления данных;
2. оценить вычислительные ограничения локальной обработки;
3. сравнить три режима обработки: batch, micro-batch и stream;
4. сформировать архитектурную рекомендацию;
5. подготовить инженерное обоснование выбора.

## Что считается результатом

Студент должен представить:
- выполненный ноутбук;
- таблицы и графики;
- текстовое архитектурное заключение;
- ответы на индивидуальные вопросы своего варианта.


In [ ]:

# Импорт библиотек
import os
import time
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)



## Шаг 1. Выбор варианта

Укажите номер своего варианта.  
Номер варианта задаёт:
- предметную область;
- индивидуальный датасет;
- SLA (допустимую задержку);
- фокус инженерного анализа.


In [ ]:

VARIANT = 1  # <-- замените на свой номер от 1 до 15

catalog = pd.read_csv("variant_catalog.csv")
variant_row = catalog.loc[catalog["variant"] == VARIANT].iloc[0]
variant_row



### Интерпретация варианта

Внимательно прочитайте:
- тему;
- описание набора событий;
- региональный контекст;
- SLA;
- индивидуальный вопрос.

Именно эти параметры должны учитываться в ваших выводах.


In [ ]:

dataset_file = variant_row["dataset_file"]
df = pd.read_csv(dataset_file, parse_dates=["event_time"])

print(f"Файл варианта: {dataset_file}")
print(f"Количество строк: {len(df):,}")
print(f"Количество столбцов: {df.shape[1]}")
display(df.head())



## Шаг 2. Первичный технический профиль датасета

На этом шаге нужно оценить:
- размер файла;
- объём данных в памяти;
- типы столбцов;
- пропуски;
- дубликаты;
- базовый состав событий.

### Задание студенту
1. Зафиксируйте размер файла и объём памяти DataFrame.
2. Определите потенциальные проблемы для локальной среды.
3. Сделайте краткий вывод: можно ли уверенно анализировать этот набор на одном ноутбуке без оптимизации?


In [ ]:

file_size_mb = Path(dataset_file).stat().st_size / 1024**2
memory_mb = df.memory_usage(deep=True).sum() / 1024**2

profile_table = pd.DataFrame({
    "Показатель": ["Размер CSV, МБ", "Объём DataFrame в памяти, МБ", "Число строк", "Число столбцов", "Дубликаты, шт.", "Пропуски, шт."],
    "Значение": [round(file_size_mb, 2), round(memory_mb, 2), len(df), df.shape[1], int(df.duplicated().sum()), int(df.isna().sum().sum())]
})
display(profile_table)

display(df.dtypes.to_frame("dtype"))
display(df.isna().sum().to_frame("missing"))



## Шаг 3. Профиль нагрузки во времени

Для архитектурного выбора нужно понять:
- насколько равномерно поступают события;
- есть ли пики;
- какой максимальный поток наблюдается;
- каков профиль нагрузки по часам и по дням.

### Задание студенту
1. Постройте агрегаты по часам и по дням.
2. Выявите пиковые интервалы.
3. Объясните, что означают найденные пики для архитектуры обработки.


In [ ]:

df = df.sort_values("event_time").reset_index(drop=True)
hourly = df.groupby(pd.Grouper(key="event_time", freq="H")).size().rename("events_per_hour").reset_index()
daily = df.groupby(pd.Grouper(key="event_time", freq="D")).size().rename("events_per_day").reset_index()
per_minute = df.groupby(pd.Grouper(key="event_time", freq="min")).size().rename("events_per_min").reset_index()

peak_hour = hourly.loc[hourly["events_per_hour"].idxmax()]
peak_min = per_minute.loc[per_minute["events_per_min"].idxmax()]

display(hourly.head())
display(daily.head())

print("Пиковый час:")
display(peak_hour.to_frame().T)
print("Пиковая минута:")
display(peak_min.to_frame().T)


In [ ]:

plt.figure(figsize=(10,4))
plt.plot(hourly["event_time"], hourly["events_per_hour"])
plt.title(f"Вариант {VARIANT}: поток событий по часам")
plt.xlabel("Время")
plt.ylabel("Событий в час")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,4))
plt.bar(daily["event_time"].dt.strftime("%Y-%m-%d"), daily["events_per_day"])
plt.title(f"Вариант {VARIANT}: поток событий по дням")
plt.xlabel("Дата")
plt.ylabel("Событий в день")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()



## Шаг 4. Оценка роста масштаба

Реальный инженерный выбор нельзя делать только по текущему объёму.  
Нужно оценить, что произойдёт при росте нагрузки в 10, 50 и 100 раз.

### Задание студенту
1. Экстраполируйте объём файла и памяти.
2. Оцените, что станет узким местом при росте.
3. Сформулируйте критерий перехода от локальной обработки к более серьёзной архитектуре.


In [ ]:

scale_factors = [1, 10, 50, 100]
growth = pd.DataFrame({
    "scale_factor": scale_factors,
    "rows_estimated": [len(df) * k for k in scale_factors],
    "file_size_mb_estimated": [round(file_size_mb * k, 2) for k in scale_factors],
    "memory_mb_estimated": [round(memory_mb * k, 2) for k in scale_factors],
    "peak_events_per_hour_estimated": [int(peak_hour["events_per_hour"] * k) for k in scale_factors],
})
display(growth)



## Шаг 5. Сравнение режимов обработки

Мы сравним три режима:

### 5.1. Batch
Полный пересчёт завершённого массива.  
Подходит для отчётности, ретроспективной аналитики, ночных перерасчётов.

### 5.2. Micro-batch
Обработка маленькими пакетами через короткие интервалы.  
Компромисс между полнотой и задержкой.

### 5.3. Stream
Почти непрерывная обработка событий по мере поступления.  
Нужна там, где важна минимальная задержка.

### Задание студенту
По результатам анализа вы должны определить:
- какой режим является основным;
- какой может быть вспомогательным;
- нужен ли гибридный контур.


In [ ]:

# Небольшая симуляция времени выполнения на локальной машине

def timed_batch_aggregation(frame):
    t0 = time.perf_counter()
    result = frame.groupby(["event_type", "priority"]).agg(
        events=("event_type", "size"),
        total_volume=("metric_volume", "sum"),
        total_value=("metric_value", "sum"),
        avg_delay=("processing_delay_sec", "mean")
    ).reset_index()
    dt = time.perf_counter() - t0
    return result, dt

def timed_microbatch_aggregation(frame, window="15min"):
    t0 = time.perf_counter()
    tmp = frame.set_index("event_time")
    result = tmp.groupby([pd.Grouper(freq=window), "event_type"]).agg(
        events=("entity_id", "size"),
        total_value=("metric_value", "sum")
    ).reset_index()
    dt = time.perf_counter() - t0
    return result, dt

batch_result, batch_time = timed_batch_aggregation(df.copy())
micro_result, micro_time = timed_microbatch_aggregation(df.copy())

print(f"Batch aggregate time:      {batch_time:.4f} sec")
print(f"Micro-batch aggregate time:{micro_time:.4f} sec")
display(batch_result.head())
display(micro_result.head())



## Шаг 6. Псевдопотоковая модель

В учебной среде мы не поднимаем полноценную потоковую платформу, но можем оценить:
- долю событий, требующих реакции быстрее SLA;
- насколько часто возникают пиковые интервалы;
- сколько данных должно проходить через потоковый контур.

### Задание студенту
1. Определите долю событий с высоким приоритетом.
2. Оцените, какой процент событий нарушает SLA по задержке.
3. Сделайте вывод: нужен ли отдельный потоковый контур или достаточно микропакетов.


In [ ]:

priority_share = (df["priority"] == "high").mean()
sla_seconds = int(variant_row["sla_seconds"])
sla_breach_share = (df["processing_delay_sec"] > sla_seconds).mean()

stream_need = pd.DataFrame({
    "Показатель": ["Доля high-priority событий", "Доля событий с нарушением SLA", "SLA, сек", "Пиковые события в минуту"],
    "Значение": [round(priority_share, 4), round(sla_breach_share, 4), sla_seconds, int(peak_min["events_per_min"])]
})
display(stream_need)



## Шаг 7. Инженерный скоринг архитектуры

Ниже используется **учебная эвристика**, а не отраслевой стандарт.  
Её задача — помочь вам логически связать свойства данных и архитектурное решение.

### Логика скоринга
- если SLA жёсткое и доля high-priority событий велика, score stream растёт;
- если пиковая нагрузка умеренная, но требуется регулярное обновление, растёт score micro-batch;
- если задержка допускается и нужен полный пересчёт, растёт score batch.

Вы обязаны:
1. посчитать score;
2. не ограничиться числом;
3. дать содержательное объяснение выбора.


In [ ]:

peak_hour_events = int(peak_hour["events_per_hour"])
peak_min_events = int(peak_min["events_per_min"])

batch_score = 0
microbatch_score = 0
stream_score = 0

# SLA
if sla_seconds >= 300:
    batch_score += 3
    microbatch_score += 2
elif 60 <= sla_seconds < 300:
    microbatch_score += 3
    batch_score += 1
else:
    stream_score += 4
    microbatch_score += 2

# high priority
if priority_share > 0.12:
    stream_score += 3
elif priority_share > 0.07:
    microbatch_score += 2
else:
    batch_score += 1

# SLA breaches
if sla_breach_share > 0.15:
    stream_score += 3
    microbatch_score += 1
elif sla_breach_share > 0.05:
    microbatch_score += 2
else:
    batch_score += 1

# peak intensity
if peak_min_events > 25:
    stream_score += 3
elif peak_min_events > 10:
    microbatch_score += 2
else:
    batch_score += 1

# data growth
if growth.loc[growth["scale_factor"] == 100, "memory_mb_estimated"].iloc[0] > 2000:
    microbatch_score += 2
    stream_score += 1
else:
    batch_score += 1

scores = pd.DataFrame({
    "architecture": ["batch", "micro_batch", "stream"],
    "score": [batch_score, microbatch_score, stream_score]
}).sort_values("score", ascending=False)
display(scores)

recommended = scores.iloc[0]["architecture"]
print("Рекомендуемая архитектура по учебной эвристике:", recommended)



## Шаг 8. Таблица архитектурного решения

Заполните и интерпретируйте следующую таблицу:
- основной режим обработки;
- вспомогательный режим;
- обоснование;
- риск;
- условия перехода к распределённой среде.

### Важно
Здесь оценивается **инженерная аргументация**, а не совпадение с «правильным ответом».


In [ ]:

decision_table = pd.DataFrame({
    "Критерий": [
        "Основной режим обработки",
        "Вспомогательный режим",
        "Главный фактор выбора",
        "Основной риск",
        "Когда локальная среда перестанет быть достаточной",
        "Нужен ли отдельный потоковый контур",
        "Нужен ли ночной batch-пересчёт"
    ],
    "Ваш вывод": [
        "",
        "",
        "",
        "",
        "",
        "",
        ""
    ]
})
decision_table



## Шаг 9. Индивидуальные задания по вариантам

Ниже приведены вопросы, на которые необходимо ответить **именно по вашему варианту**.

### Обязательные ответы
1. Какой класс событий в вашем наборе требует минимальной задержки?
2. Какие показатели следует считать потоково, а какие пакетно?
3. Что станет первым узким местом при росте нагрузки в 100 раз?
4. Достаточна ли micro-batch архитектура или уже нужен stream-контур?
5. Какие компоненты будущего пайплайна вы бы выделили: ingest, storage, processing, serving, monitoring?

Ответы оформите текстом в следующей ячейке.


In [ ]:

individual_answers = {
    "variant": int(variant_row["variant"]),
    "theme": variant_row["theme"],
    "focus_question": variant_row["focus_question"],
    "answer_1_min_latency_events": "",
    "answer_2_stream_vs_batch_metrics": "",
    "answer_3_first_bottleneck_x100": "",
    "answer_4_microbatch_or_stream": "",
    "answer_5_pipeline_components": ""
}
individual_answers



## Шаг 10. Итоговое архитектурное заключение

Подготовьте короткое инженерное заключение объёмом 8-12 предложений.  
В нём должны присутствовать:
- характеристика нагрузки;
- оценка SLA;
- вывод о достаточности локальной обработки;
- основной и вспомогательный режимы;
- условия перехода к распределённой архитектуре.

Ниже предусмотрена шаблонная структура.


In [ ]:

conclusion_template = f'''
Вариант {VARIANT}: {variant_row["theme"]}.
Набор данных отражает ............................................................
Пиковая нагрузка составляет ......................................................
SLA по варианту равно ............................................................
Поэтому в качестве основного режима обработки предлагается .......................
Вспомогательным режимом целесообразно считать ....................................
Локальная среда достаточна / недостаточна при условиях ...........................
При росте в 10-100 раз узким местом станет .......................................
Для промышленного решения следует выделить контуры ...............................
'''
print(conclusion_template)



## Шаг 11. Экспорт мини-отчёта

Сохраните ключевые результаты в CSV/JSON, чтобы можно было приложить их к отчёту.


In [ ]:

scores.to_csv(f"pz8_variant_{VARIANT:02d}_scores.csv", index=False, encoding="utf-8-sig")
growth.to_csv(f"pz8_variant_{VARIANT:02d}_growth.csv", index=False, encoding="utf-8-sig")

with open(f"pz8_variant_{VARIANT:02d}_answers.json", "w", encoding="utf-8") as f:
    json.dump(individual_answers, f, ensure_ascii=False, indent=2)

print("Файлы мини-отчёта сохранены.")



# Контрольные выводы студента

Перед сдачей убедитесь, что вы:
- указали свой вариант;
- выполнили все вычисления;
- заполнили таблицу архитектурного решения;
- ответили на индивидуальные вопросы;
- сформулировали итоговое заключение;
- сохранили мини-отчёт.

---

## Что оценивается преподавателем
1. Корректность вычислений.
2. Качество интерпретации.
3. Связь между профилем нагрузки и архитектурным выбором.
4. Умение различать batch, micro-batch и stream.
5. Инженерная аргументация, а не только воспроизведение терминов.
